# Experiment 13 — GLCM 3×3×3 vs 11×11×11 + XGBoost, controlled comparison

This experiment keeps fixed:

- the same 577-patient MGMT cohort
- the same shared 10 folds used by Experiments 11–13
- the same 24 GLCM features
- the same patient-level `%A` representation
- the same XGBoost configuration
- the same center voxels for K1 and K5

Only local `kernelRadius` changes:

- K1: **3×3×3**
- K5: **11×11×11**

A single common center mask is used for both kernels.


## 1. Install

In [ ]:
# !pip install numpy pandas scipy SimpleITK pyradiomics scikit-learn xgboost


## 2. Imports

In [ ]:
from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
import SimpleITK as sitk
import radiomics
from scipy import ndimage as ndi
from radiomics import featureextractor

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    matthews_corrcoef, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier


## 3. Configuration

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 10
EXPECTED_PATIENTS = 577
EXPECTED_MGMT_1 = 301
EXPECTED_MGMT_0 = 276

MANIFEST_PATH = Path("/path/to/brats2021_577_manifest.csv")  # <-- change

OUTDIR = Path("outputs_exp13_glcm_k1_vs_k5")
OUTDIR.mkdir(parents=True, exist_ok=True)

SHARED_DIR = Path("shared_rao_comparison")
SHARED_FOLD_CSV = SHARED_DIR / "mgmt_577_fold_assignment_10fold.csv"

KERNELS = {
    "K1_3x3x3": 1,
    "K5_11x11x11": 5,
}
COMMON_MARGIN = max(KERNELS.values())

GLCM_FEATURES = [
    "Autocorrelation","ClusterProminence","ClusterShade","ClusterTendency",
    "Contrast","Correlation","DifferenceAverage","DifferenceEntropy",
    "DifferenceVariance","Id","Idm","Idmn","Idn","Imc1","Imc2",
    "InverseVariance","JointAverage","JointEnergy","JointEntropy","MCC",
    "MaximumProbability","SumAverage","SumEntropy","SumSquares",
]
assert len(GLCM_FEATURES) == 24

PYRAD_SETTINGS = {
    "binWidth": 25,
    "normalize": False,
    "resampledPixelSpacing": None,
    "force2D": False,
    "distances": [1],
    "maskedKernel": False,
}

WEIGHT_CONDITIONS = ["None", "Balanced"]
REUSE_CACHE = True

print("Installed PyRadiomics:", radiomics.__version__)
print("PyRadiomics settings:", PYRAD_SETTINGS)


## 4. Load cohort

In [ ]:
def load_manifest(path):
    df = pd.read_csv(path)
    req = {"patient_id", "flair_path", "seg_path", "mgmt"}
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df["patient_id"] = df["patient_id"].astype(str)
    df["mgmt"] = df["mgmt"].astype(int)
    df = df.sort_values("patient_id").reset_index(drop=True)

    if df["patient_id"].duplicated().any():
        raise ValueError("Duplicate patient_id values found.")
    if len(df) != 577 or df["patient_id"].nunique() != 577:
        raise ValueError("Experiment 13 requires exactly 577 unique patients.")

    counts = df["mgmt"].value_counts().to_dict()
    if counts.get(1, 0) != EXPECTED_MGMT_1 or counts.get(0, 0) != EXPECTED_MGMT_0:
        raise ValueError(f"Unexpected MGMT distribution: {counts}")

    for c in ["flair_path", "seg_path"]:
        df[c] = df[c].map(lambda x: str(Path(x).expanduser().resolve()))
        missing_files = [p for p in df[c] if not Path(p).exists()]
        if missing_files:
            raise FileNotFoundError(f"{c}: missing file, e.g. {missing_files[0]}")
    return df


def stable_hash(payload):
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def cohort_signature(df):
    return [
        {
            "patient_id": str(r.patient_id),
            "mgmt": int(r.mgmt),
            "flair_path": str(r.flair_path),
            "seg_path": str(r.seg_path),
        }
        for r in df.sort_values("patient_id").itertuples(index=False)
    ]


manifest = load_manifest(MANIFEST_PATH)
print("Patients:", len(manifest))
print(manifest["mgmt"].value_counts().sort_index())
display(manifest.head())


## 5. Prepare the Rao 33×33×33 tumor crop

Rao uses two different spatial scales:

- **33×33×33**: standardized whole-tumor crop
- **3×3×3**: local K1 radiomics neighborhood

Therefore, the crop must not be changed from 33×33×33 to 3×3×3. Doing so would collapse the entire standardized tumor volume into one local cube and would no longer reproduce Rao's method.

The code below keeps the 33×33×33 crop and corrects the resampling spacing so physical extent is preserved.


In [ ]:
def bbox_from_mask(mask):
    coords = np.argwhere(mask)
    if not len(coords):
        raise ValueError("Empty tumor mask")
    z0,y0,x0 = coords.min(axis=0)
    z1,y1,x1 = coords.max(axis=0) + 1
    return (x0,y0,z0), (x1,y1,z1)


def cube_bounds(mask):
    (x0,y0,z0),(x1,y1,z1)=bbox_from_mask(mask)
    side=max(x1-x0,y1-y0,z1-z0)
    cx=(x0+x1-1)/2
    cy=(y0+y1-1)/2
    cz=(z0+z1-1)/2

    starts=[
        int(round(cx-(side-1)/2)),
        int(round(cy-(side-1)/2)),
        int(round(cz-(side-1)/2)),
    ]
    dims=[mask.shape[2],mask.shape[1],mask.shape[0]]

    for d in range(3):
        starts[d]=max(0,starts[d])
        if starts[d]+side>dims[d]:
            starts[d]=max(0,dims[d]-side)

    size=[min(side,dims[d]-starts[d]) for d in range(3)]
    return tuple(starts),tuple(size)


def crop_sitk(image,start,size):
    return sitk.RegionOfInterest(
        image,
        size=[int(v) for v in size],
        index=[int(v) for v in start],
    )


def resample33(image,is_mask=False):
    old_size=np.array(image.GetSize(),float)
    old_spacing=np.array(image.GetSpacing(),float)
    new_size=np.array([33,33,33],int)

    new_spacing=old_spacing*np.maximum(old_size-1,1)/np.maximum(new_size-1,1)

    r=sitk.ResampleImageFilter()
    r.SetSize([int(v) for v in new_size])
    r.SetOutputSpacing([float(v) for v in new_spacing])
    r.SetOutputOrigin(image.GetOrigin())
    r.SetOutputDirection(image.GetDirection())
    r.SetTransform(sitk.Transform())
    r.SetDefaultPixelValue(0)
    r.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)

    pixel_type=sitk.sitkUInt8 if is_mask else sitk.sitkFloat32
    return r.Execute(sitk.Cast(image,pixel_type))


def prepare33(flair_path,seg_path):
    flair=sitk.ReadImage(str(flair_path))
    seg=sitk.ReadImage(str(seg_path))

    tumor=sitk.GetArrayFromImage(seg)!=0
    start,size=cube_bounds(tumor)

    fc=crop_sitk(flair,start,size)
    sc=crop_sitk(seg,start,size)

    mask_arr=(sitk.GetArrayFromImage(sc)!=0).astype(np.uint8)
    mask=sitk.GetImageFromArray(mask_arr)
    mask.CopyInformation(sc)

    return resample33(fc,False),resample33(mask,True)


## 6. Global GLCM medians for activation thresholds

In [ ]:
def build_global_glcm_extractor():
    ex=featureextractor.RadiomicsFeatureExtractor(**PYRAD_SETTINGS)
    ex.disableAllFeatures()
    ex.enableFeaturesByName(glcm=GLCM_FEATURES)
    ex.disableAllImageTypes()
    ex.enableImageTypeByName("Original")
    return ex


def extract_global_glcm(manifest):
    out=OUTDIR/"global_glcm_24.csv"
    meta=OUTDIR/"global_glcm_24.meta.json"

    payload={
        "stage":"global_glcm_24",
        "cohort":cohort_signature(manifest),
        "pyradiomics_version":radiomics.__version__,
        "settings":PYRAD_SETTINGS,
        "features":GLCM_FEATURES,
    }
    h=stable_hash(payload)

    if REUSE_CACHE and out.exists() and meta.exists():
        m=json.loads(meta.read_text(encoding="utf-8"))
        if m.get("config_hash")==h:
            print("Reusing validated cache:",out)
            return pd.read_csv(out)
        print("Global cache hash mismatch; recomputing.")

    ex=build_global_glcm_extractor()
    rows=[]

    for i,row in manifest.iterrows():
        print(f"[Global {i+1}/{len(manifest)}] {row.patient_id}")
        flair=sitk.ReadImage(row.flair_path)
        seg=sitk.ReadImage(row.seg_path)

        arr=(sitk.GetArrayFromImage(seg)!=0).astype(np.uint8)
        mask=sitk.GetImageFromArray(arr)
        mask.CopyInformation(seg)

        res=ex.execute(flair,mask)
        rec={"patient_id":row.patient_id,"mgmt":int(row.mgmt)}

        for k,v in res.items():
            if k.startswith("original_glcm_"):
                rec[k.replace("original_glcm_","")]=float(v)
        rows.append(rec)

    df=pd.DataFrame(rows)

    missing=sorted(set(GLCM_FEATURES)-set(df.columns))
    if missing:
        raise RuntimeError(f"Missing GLCM features: {missing}")
    if len(df)!=577:
        raise RuntimeError("Expected 577 global rows.")

    df.to_csv(out,index=False)
    meta.write_text(
        json.dumps({"config_hash":h,"payload":payload},indent=2,ensure_ascii=False),
        encoding="utf-8",
    )
    return df


global_glcm=extract_global_glcm(manifest)
global_medians={c:float(global_glcm[c].median()) for c in GLCM_FEATURES}
display(global_glcm.head())


## 7. Local GLCM maps and patient-level `%A`

In [ ]:
def common_center_mask(tumor_img):
    # Rao eligibility: at least one tumor voxel in the local 3×3×3 cube.
    arr=sitk.GetArrayFromImage(tumor_img).astype(np.uint8)

    count3=ndi.convolve(
        arr,
        np.ones((3,3,3),dtype=np.uint8),
        mode="constant",
        cval=0,
    )
    centers=(count3>=1)

    # Same selected centers must also support a full 11×11×11 K5 neighborhood.
    m=COMMON_MARGIN
    centers[:m,:,:]=False
    centers[-m:,:,:]=False
    centers[:,:m,:]=False
    centers[:,-m:,:]=False
    centers[:,:,:m]=False
    centers[:,:,-m:]=False

    out=sitk.GetImageFromArray(centers.astype(np.uint8))
    out.CopyInformation(tumor_img)
    return out


def build_local_glcm_extractor(radius):
    settings=dict(PYRAD_SETTINGS)
    settings["kernelRadius"]=radius
    ex=featureextractor.RadiomicsFeatureExtractor(**settings)
    ex.disableAllFeatures()
    ex.enableFeaturesByName(glcm=GLCM_FEATURES)
    ex.disableAllImageTypes()
    ex.enableImageTypeByName("Original")
    return ex


def map_values_at_mask(fmap,mask):
    aligned=sitk.Resample(
        mask,
        fmap,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        sitk.sitkUInt8,
    )
    f=sitk.GetArrayFromImage(fmap)
    m=sitk.GetArrayFromImage(aligned).astype(bool)
    return np.asarray(f[m],dtype=float)


def extract_activation_for_kernel(manifest,label,radius):
    out=OUTDIR/f"local_glcm_activation_{label}.csv"
    meta=OUTDIR/f"local_glcm_activation_{label}.meta.json"

    payload={
        "stage":"local_glcm_activation",
        "label":label,
        "radius":radius,
        "cohort":cohort_signature(manifest),
        "pyradiomics_version":radiomics.__version__,
        "settings":PYRAD_SETTINGS,
        "features":GLCM_FEATURES,
        "crop_size":[33,33,33],
        "common_center_rule":"3x3x3 contains >=1 tumor voxel; border margin=5",
        "common_margin":COMMON_MARGIN,
    }
    h=stable_hash(payload)

    if REUSE_CACHE and out.exists() and meta.exists():
        m=json.loads(meta.read_text(encoding="utf-8"))
        if m.get("config_hash")==h:
            print("Reusing validated cache:",out)
            return pd.read_csv(out)
        print(label,"cache hash mismatch; recomputing.")

    ex=build_local_glcm_extractor(radius)
    rows=[]

    for i,row in manifest.iterrows():
        print(f"[{label} {i+1}/{len(manifest)}] {row.patient_id}")

        flair33,tumor33=prepare33(row.flair_path,row.seg_path)
        centers=common_center_mask(tumor33)

        n_centers=int(sitk.GetArrayFromImage(centers).sum())
        if n_centers==0:
            raise RuntimeError(f"No common centers for patient {row.patient_id}")

        res=ex.execute(flair33,centers,label=1,voxelBased=True)

        rec={
            "patient_id":row.patient_id,
            "mgmt":int(row.mgmt),
            "n_common_centers":n_centers,
        }

        for k,v in res.items():
            if not (k.startswith("original_glcm_") and isinstance(v,sitk.Image)):
                continue

            feat=k.replace("original_glcm_","")
            vals=map_values_at_mask(v,centers)
            vals=vals[np.isfinite(vals)]

            rec[feat]=np.nan if len(vals)==0 else float(
                np.mean(vals>=global_medians[feat])
            )

        rows.append(rec)

    df=pd.DataFrame(rows)

    if len(df)!=577 or df["patient_id"].nunique()!=577:
        raise RuntimeError("Local activation table must contain exactly 577 patients.")

    missing=sorted(set(GLCM_FEATURES)-set(df.columns))
    if missing:
        raise RuntimeError(f"Missing local GLCM features: {missing}")

    bad=~np.isfinite(df[GLCM_FEATURES].to_numpy(float))
    if bad.any():
        bad_patients=df.loc[bad.any(axis=1),"patient_id"].tolist()
        raise RuntimeError(
            "NaN/Inf detected; cohort reduction is forbidden. "
            f"Affected patients: {bad_patients[:10]}"
        )

    df.to_csv(out,index=False)
    meta.write_text(
        json.dumps({"config_hash":h,"payload":payload},indent=2,ensure_ascii=False),
        encoding="utf-8",
    )
    return df


activation={}
for label,radius in KERNELS.items():
    activation[label]=extract_activation_for_kernel(manifest,label,radius)
    print(label,activation[label].shape)

center_check=(
    activation["K1_3x3x3"][["patient_id","n_common_centers"]]
    .merge(
        activation["K5_11x11x11"][["patient_id","n_common_centers"]],
        on="patient_id",
        suffixes=("_K1","_K5"),
        validate="one_to_one",
    )
)

if not (
    center_check["n_common_centers_K1"]
    == center_check["n_common_centers_K5"]
).all():
    raise RuntimeError("K1 and K5 did not use identical common center counts.")


## 8. Create/reuse identical folds

In [ ]:
def load_shared_folds(cohort):
    if not SHARED_FOLD_CSV.exists():
        raise FileNotFoundError(
            f"{SHARED_FOLD_CSV} not found. Run Experiment 11 first."
        )

    folds=pd.read_csv(SHARED_FOLD_CSV)
    folds["patient_id"]=folds["patient_id"].astype(str)

    if len(folds)!=577 or folds["patient_id"].nunique()!=577:
        raise ValueError("Shared fold file must contain 577 unique patients.")

    expected=cohort[["patient_id","mgmt"]]
    merged=expected.merge(
        folds[["patient_id","mgmt","fold"]],
        on=["patient_id","mgmt"],
        how="outer",
        indicator=True,
    )

    if not (merged["_merge"]=="both").all():
        raise ValueError("Shared folds do not match this cohort.")

    return folds[["patient_id","fold"]]


folds=load_shared_folds(manifest)
print(
    manifest.merge(folds,on="patient_id")
    .groupby(["fold","mgmt"]).size().unstack(fill_value=0)
)


## 9. XGBoost evaluation

In [ ]:
def build_xgb():
    return XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        min_child_weight=1,
        gamma=0,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0,
        reg_lambda=1,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def specificity(y,p):
    tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel()
    return tn/(tn+fp) if tn+fp else np.nan


rows=[]

for kernel_label,df in activation.items():
    work=df.merge(folds,on="patient_id",how="inner",validate="one_to_one")

    if len(work)!=577:
        raise RuntimeError(f"{kernel_label}: expected 577 patients after fold merge.")

    for weight_condition in WEIGHT_CONDITIONS:
        for fold in range(1,N_SPLITS+1):
            tr=work[work.fold!=fold]
            va=work[work.fold==fold]

            Xtr=tr[GLCM_FEATURES].to_numpy(np.float32)
            ytr=tr.mgmt.to_numpy(int)
            Xva=va[GLCM_FEATURES].to_numpy(np.float32)
            yva=va.mgmt.to_numpy(int)

            sample_weight=None
            if weight_condition=="Balanced":
                classes=np.array([0,1])
                cwv=compute_class_weight("balanced",classes=classes,y=ytr)
                cw={0:float(cwv[0]),1:float(cwv[1])}
                sample_weight=np.array(
                    [cw[int(v)] for v in ytr],
                    dtype=np.float32,
                )

            m=build_xgb()
            m.fit(Xtr,ytr,sample_weight=sample_weight)

            prob=m.predict_proba(Xva)[:,1]
            pred=(prob>=0.5).astype(int)

            rows.append({
                "kernel":kernel_label,
                "condition":weight_condition,
                "fold":fold,
                "accuracy":accuracy_score(yva,pred),
                "precision":precision_score(yva,pred,zero_division=0),
                "recall":recall_score(yva,pred,zero_division=0),
                "specificity":specificity(yva,pred),
                "f1":f1_score(yva,pred,zero_division=0),
                "roc_auc":roc_auc_score(yva,prob),
                "pr_auc":average_precision_score(yva,prob),
                "balanced_accuracy":balanced_accuracy_score(yva,pred),
                "mcc":matthews_corrcoef(yva,pred),
            })

per_fold=pd.DataFrame(rows)
per_fold.to_csv(OUTDIR/"exp13_k1_vs_k5_per_fold.csv",index=False)

metric_cols=[
    "accuracy","precision","recall","specificity","f1",
    "roc_auc","pr_auc","balanced_accuracy","mcc",
]

summary=(
    per_fold
    .groupby(["kernel","condition"])[metric_cols]
    .agg(["mean","std"])
)

display(summary)
summary.to_csv(OUTDIR/"exp13_k1_vs_k5_summary.csv")


## 10. Interpretation

This is now a controlled K1-vs-K5 comparison:

- same 577 patients
- same shared 10 folds
- same 24 GLCM features
- same XGBoost
- same `%A` representation
- same common center voxels

Only `kernelRadius` changes from 1 (3×3×3) to 5 (11×11×11).

`None` is the primary Rao-oriented condition; `Balanced` is a supplementary thesis-oriented condition.

### Spatial-scale note

The standardized whole-tumor crop remains **33×33×33**.  
The **3×3×3** value refers to the local K1 neighborhood. Replacing the entire 33×33×33 crop with 3×3×3 would no longer follow Rao's method.
